# 01. Exploración de la oferta programada de TransMilenio

Este notebook inicia el proyecto con una pregunta pequeña: **¿qué rutas troncales y cuántos viajes aparecen en el horario publicado?**

Trabajamos con el GTFS estático. Por ahora **no medimos buses reales, retrasos, ocupación ni demanda**. Más adelante contrastaremos la oferta programada por estación y franja con las validaciones de pasajeros, cuando esa fuente esté descargada y revisada.

## Antes de ejecutar
1. Extrae de `GTFS.zip` los archivos `agency.txt`, `routes.txt`, `trips.txt` y `stops.txt` sin modificarlos.
2. Colócalos en `datos/raw/gtfs/`, dentro de este repositorio. Esa carpeta está excluida de Git: cada integrante debe tener los archivos en su equipo.
3. Abre el proyecto completo en VS Code y ejecuta las celdas en orden.

## 1. Librerías y rutas

In [ ]:
import pandas as pd
from pathlib import Path

# Si VS Code inicia dentro de notebooks/, subimos a la carpeta del proyecto.
carpeta_proyecto = Path.cwd()
if carpeta_proyecto.name == 'notebooks':
    carpeta_proyecto = carpeta_proyecto.parent

carpeta_gtfs = carpeta_proyecto / 'datos' / 'raw' / 'gtfs'
print('Buscando las tablas en:', carpeta_gtfs)

# Comprobar los archivos antes de cargar las tablas.
for archivo in ['agency.txt', 'routes.txt', 'trips.txt', 'stops.txt']:
    if not (carpeta_gtfs / archivo).exists():
        raise FileNotFoundError(f'Falta {archivo} en {carpeta_gtfs}. Extrae las cuatro tablas del GTFS.zip en esa carpeta.')

## 2. Carga de las tablas pequeñas
`stop_times.txt` es mucho más grande; todavía no lo cargamos para cuidar la memoria del computador.

In [ ]:
# Cada tabla responde una pregunta distinta: agencias, rutas, viajes y paradas.
agencias = pd.read_csv(carpeta_gtfs / 'agency.txt')
rutas = pd.read_csv(carpeta_gtfs / 'routes.txt')
viajes = pd.read_csv(carpeta_gtfs / 'trips.txt')
paradas = pd.read_csv(carpeta_gtfs / 'stops.txt')

print('Agencias:', agencias.shape)
print('Rutas:', rutas.shape)
print('Viajes programados:', viajes.shape)
print('Paradas:', paradas.shape)
display(agencias[['agency_id', 'agency_name']])

## 3. Comprobaciones básicas de calidad
Revisamos si faltan las llaves que necesitamos y si hay identificadores repetidos.

In [ ]:
print('Rutas sin ID:', rutas['route_id'].isna().sum())
print('IDs de ruta repetidos:', rutas['route_id'].duplicated().sum())
print('Viajes sin ID:', viajes['trip_id'].isna().sum())
print('IDs de viaje repetidos:', viajes['trip_id'].duplicated().sum())
print('Viajes sin ruta:', viajes['route_id'].isna().sum())
print('Paradas sin ID:', paradas['stop_id'].isna().sum())

## 4. Filtrar el componente troncal
En este GTFS, la agencia 1 aparece como `Transmilenio-Troncal`. Verificamos su nombre antes de utilizar el filtro.

In [ ]:
# El filtro se apoya en el nombre que acabamos de mostrar en la tabla agencias.
agencia_troncal = agencias[agencias['agency_name'] == 'Transmilenio-Troncal']
print('Filas encontradas para la agencia troncal:', len(agencia_troncal))

if len(agencia_troncal) == 1:
    id_agencia_troncal = agencia_troncal['agency_id'].iloc[0]
    rutas_troncales = rutas[rutas['agency_id'] == id_agencia_troncal].copy()
    viajes_troncales = viajes[viajes['route_id'].isin(rutas_troncales['route_id'])].copy()
    print('Rutas troncales:', len(rutas_troncales))
    print('Viajes troncales en todo el GTFS:', len(viajes_troncales))
else:
    print('Revisar agency.txt: no se encontró una única agencia troncal.')

## 5. Primer resumen por ruta
Contamos los viajes publicados por ruta y por tipo de servicio. **No son buses que hayan circulado realmente**, ni viajes de una fecha concreta: todavía falta aplicar el calendario.

In [ ]:
# Contar filas de trip_id dentro de cada ruta y servicio.
viajes_por_ruta = viajes_troncales.groupby(['route_id', 'service_id'])['trip_id'].count().reset_index()
viajes_por_ruta = viajes_por_ruta.rename(columns={'trip_id': 'viajes_programados'})

# Añadir el código y el destino de cada ruta para que la tabla sea entendible.
resumen_oferta = pd.merge(
    viajes_por_ruta,
    rutas_troncales[['route_id', 'route_short_name', 'route_long_name']],
    on='route_id',
    how='left'
)
resumen_oferta = resumen_oferta.sort_values('viajes_programados', ascending=False)
display(resumen_oferta.head(15))

## 6. Lo que este resultado permite y no permite decir

- Permite conocer las rutas y viajes que aparecen en la programación publicada.
- Todavía **no** indica cuántos viajes corresponden a un día concreto: para eso leeremos `calendar.txt` y `calendar_dates.txt`.
- Todavía **no** indica horarios por estación: para eso necesitaremos `stop_times.txt`, leído de forma que no agote la memoria.
- Todavía **no** compara oferta con demanda: faltan las validaciones diarias.
- Una posición de Paradero o de `positions.pb` no equivale a un viaje cumplido ni a una llegada observada.